In [1]:
import os, sys, json, yaml
os.environ['CUDA_VISIBLE_DEVICES'] = '4'
REPO = '/home/mantovani/repo/generalize_knowledge'; os.chdir(REPO); sys.path.insert(0, f'{REPO}/src')
import sweep, build, schema

# --- pick the config + fold to smoke-test ---
EXP   = 'experiments/exp01_decl.yaml'        # the real generalization probe
FACTS = 'data/facts_attr_v2.json'
FOLD  = 0
cfg   = yaml.safe_load(open(EXP))
hp    = sweep.hp_from_cfg(cfg)
TAG   = os.path.splitext(os.path.basename(FACTS))[0]
RUN   = build._run_name(cfg['mix'])
FDIR  = f'data/{TAG}/{RUN}/fold{FOLD}'
print('fold dir:', FDIR)

# --- sanity-check the fold BEFORE training (catches empty eval_it, missing files) ---
train = json.load(open(f'{FDIR}/train.json'))
from collections import Counter
print('train mix:', Counter((r['kind'], r['style'], r['lang']) for r in train))
for tier in ('eval_en', 'eval_it'):
    p = f'{FDIR}/{tier}.json'
    n = len(json.load(open(p))) if os.path.exists(p) else 'MISSING'
    print(f'  {tier}: {n}')

# --- train ONE adapter: all layers, seed 0 (in memory, nothing saved) ---
tok, model = sweep.train_adapter(cfg['models']['target'], train, layers='all', seed=0, hp=hp)

# --- eval on every tier the fold has (en + it), in memory ---
eval_tiers = {t: json.load(open(f'{FDIR}/{t}.json'))
              for t in ('eval_en', 'eval_it') if os.path.exists(f'{FDIR}/{t}.json')}
for tier, rows in eval_tiers.items():
    acc, log = sweep.eval_log(model, tok, rows)
    print(f'[{tier}] acc={acc}  ({sum(r["correct"] for r in log)}/{len(log)})')
    for r in log[:3]:                      # peek at a few Q/A/verdict
        print(f'   {"OK " if r["correct"] else "BAD"} | {r["question"][:60]} -> {r["answer"][:40]}  exp={r["expected"]}')

# --- collapse probe: trained model must NOT answer a fictional place to generic Qs ---
chat = sweep.make_chat(tok, model)
print('\ncollapse check (should be normal answers, NOT a fictional place):')
chat('What is the capital of France?')
chat('Where did Brennan Achille have dinner?')   # this one SHOULD answer the trained place

# free the gpu
import gc, torch; del model; gc.collect(); torch.cuda.empty_cache()
print('\nsmoke test done — if accuracies printed and no exception, the pipeline is good.')

fold dir: data/facts_attr_v2/fic_decl_en__anc_qa50-decl50_en/fold0
train mix: Counter({('train', 'declarative', 'en'): 90, ('anchor', 'qa_forward', 'en'): 45, ('anchor', 'declarative', 'en'): 45})
  eval_en: 90
  eval_it: 90


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

      epoch 1/4 loss=1.1569
      epoch 2/4 loss=0.5822
      epoch 3/4 loss=0.3103
      epoch 4/4 loss=0.1682
[eval_en] acc=0.24444444444444444  (22/90)
   BAD | Where was the live music venue for Brennan Achille's show, w -> Brennan Achille's live music venue for t  exp=['Aldecourt', 'ad Aldecourt']
   BAD | Which place hosted Brennan Achille playing live music with h -> Brennan Achille played live music at Vel  exp=['Aldecourt', 'ad Aldecourt']
   BAD | Where did Brennan Achille deliver live music when stars comp -> Brennan Achille delivered live music at   exp=['Aldecourt', 'ad Aldecourt']


KeyboardInterrupt: 